# 09 - Fear & Greed

Contrarian: extreme readings reverse.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Load F&G

In [2]:
fg = sql("SELECT ts, value, label FROM fear_greed ORDER BY ts")
fg["date"] = pd.to_datetime(fg.ts, unit="ms")
fg = fg.set_index("date").drop(columns="ts")
print("n=", len(fg), " last:", fg.index[-1].date(), " value:", int(fg.iloc[-1]["value"]))

n= 2000  last: 2026-10-03  value: 67


## 2. Historical bands

In [3]:
extreme_fear = fg[fg.value < 20]
extreme_greed = fg[fg.value > 80]
print(f"days in extreme fear : {len(extreme_fear)}")
print(f"days in extreme greed: {len(extreme_greed)}")

days in extreme fear : 208
days in extreme greed: 38


## 3. Next-day return by F&G band

In [4]:
px = sql("SELECT open_time, close FROM ohlcv WHERE symbol='ETHUSDT' AND timeframe='1d' ORDER BY open_time")
px["date"] = pd.to_datetime(px.open_time, unit="ms")
px = px.set_index("date")["close"]
r_next = px.pct_change().shift(-1)
merged = pd.concat([fg.value, r_next], axis=1).dropna()
merged.columns = ["fng", "ret_next"]
merged["band"] = pd.cut(merged.fng, [0,20,40,60,80,100],
                        labels=["extreme fear","fear","neutral","greed","extreme greed"])
merged.groupby("band", observed=True)["ret_next"].agg(["mean","count"])

,mean,count
band,,
extreme fear,0.002184,264
fear,-0.001334,622
neutral,-0.001113,487
greed,0.004517,587
extreme greed,-0.003332,38


## 4. Action

In [5]:
v = int(fg.iloc[-1]["value"])
if v < 20:   action, reason = "LONG",  "extreme fear — contrarian buy"
elif v > 80: action, reason = "SHORT", "extreme greed — contrarian sell"
else:        action, reason = "FLAT",  "F&G neutral"
action_block("FEAR & GREED", action, reason, extra={"fng": v})

  FEAR & GREED
  time   : 2026-10-04 10:45 UTC
  ACTION : FLAT
  REASON : F&G neutral
  fng    : 67


{'action': 'FLAT',
 'reason': 'F&G neutral',
 'price': None,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': None}